In [7]:
# Install libraries
!pip install tensorflow gradio numpy

import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
import gradio as gr

# -----------------------------
# Sample Text Corpus
# -----------------------------
corpus = [
    "i love artificial intelligence",
    "i love machine learning",
    "deep learning is powerful",
    "artificial intelligence is the future",
    "machine learning is fun",
    "i enjoy learning new things",
    "data science is interesting",
    "ai is transforming the world",
    "learning never stops",
    "practice makes perfect"
]

# -----------------------------
# Tokenization
# -----------------------------
tokenizer = Tokenizer()
tokenizer.fit_on_texts(corpus)
total_words = len(tokenizer.word_index) + 1

# Create sequences
input_sequences = []
for line in corpus:
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram_seq = token_list[:i+1]
        input_sequences.append(n_gram_seq)

# Padding
max_seq_len = max([len(seq) for seq in input_sequences])
input_sequences = np.array(pad_sequences(input_sequences, maxlen=max_seq_len, padding='pre'))

# Split into X and y
X = input_sequences[:, :-1]
y = input_sequences[:, -1]

# One-hot encode y
y = tf.keras.utils.to_categorical(y, num_classes=total_words)

# -----------------------------
# Build LSTM Model
# -----------------------------
model = Sequential([
    Embedding(total_words, 64, input_length=max_seq_len-1),
    LSTM(100),
    Dense(total_words, activation='softmax')
])

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

print("Training model... ⏳")
model.fit(X, y, epochs=200, verbose=0)
print("Training complete ✅")

# -----------------------------
# Prediction Function
# -----------------------------
def predict_next_words(seed_text, num_words):
    results = []

    for _ in range(num_words):
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=max_seq_len-1, padding='pre')

        predicted_probs = model.predict(token_list, verbose=0)[0]

        # Get top 3 predictions
        top_indices = predicted_probs.argsort()[-3:][::-1]

        next_words = []
        for idx in top_indices:
            for word, index in tokenizer.word_index.items():
                if index == idx:
                    next_words.append(word)

        results.append(next_words)

        # Append best word to sentence
        seed_text += " " + next_words[0]

    return results

# -----------------------------
# Gradio UI Function
# -----------------------------
def autocomplete(text):
    predictions = predict_next_words(text, 3)

    output = ""
    for i, words in enumerate(predictions):
        output += f"Step {i+1}: {words}\n"

    return output

# -----------------------------
# Gradio Interface
# -----------------------------
interface = gr.Interface(
    fn=autocomplete,
    inputs=gr.Textbox(label="Enter a sentence", placeholder="Type something like: i love"),
    outputs=gr.Textbox(label="Predicted Next Words"),
    title="🧠 Text Auto-Completion using LSTM",
    description="Predict next possible words using an LSTM model. Shows top 3 suggestions."
)

interface.launch()

Training model... ⏳
Training complete ✅
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://91ebb386c0eef0dfd9.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
